In [43]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated
from pydantic import BaseModel , Field
import operator

In [44]:
load_dotenv()

True

In [45]:
model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [46]:
class EvaluationSchema(BaseModel):
    
    feedback : str = Field(description='Detailed feedback for the essay')
    score : int = Field(description="score out of 10 ", ge=0,le=10 )

In [47]:
structured_model = model.with_structured_output(EvaluationSchema)

In [56]:
essay = """Social Media

Social media is very important now days because everyone is using it every time. Peoples are always on phone and they don't do anything without social media. It is good but also bad and this is why it have many effects in the society.

First of all social media is helping people for talking with each other. If your friend is in another country you can message him very easy and fastly. Also many peoples posting photos and videos which is entertaining for everyone. Business also use social media because they can selling more products and getting much customers.

But social media also making many problems. Student waste there time by scrolling reels all day and they not studying properly. Some people become angry or sad because they seeing other peoples expensive life and they thinking there life is very bad. This create depression for many persons.

Another bad thing is fake news. Everyone can write anything on internet and peoples believe it without checking. This is making confusion and sometimes big problems in country. The companies should stop fake news but they don't doing enough.

In my opinion social media is both good and bad but mostly it depends on the peoples. If they use it carefully then it is good but if they use too much then it become dangerous. Parents should tell childrens to use phone less but many parents also using phone whole day so they cannot teach properly.

In conclusion, social media is a good invention but also bad invention. It have many advantage and disadvantage. We should use it in good way and not wasting our times because too much anything is not good for the health and future.."""

In [23]:
prompt = f'evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_output.invoke(prompt)

EvaluationSchema(feedback="The essay is well-structured and clearly articulates the dual nature of social media's impact on modern society. The language used is appropriate and the arguments are supported with relevant examples. The introduction effectively sets the stage, and the body paragraphs logically discuss both the advantages and disadvantages. The conclusion summarizes the main points and offers a forward-looking perspective on responsible usage. The essay demonstrates a good understanding of the topic and presents a balanced view. Minor improvements could include slightly more varied sentence structures to enhance flow, but overall, the language quality is strong.", score=9)

In [49]:
class UPSEState(TypedDict):
    
    essay : str
    language_feedback : str
    analysis_feedback : str
    clarity_feedback : str
    overall_feedback : str
    individual_scores : Annotated[list[int],operator.add]
    avg_score : float

In [50]:
def  evaluate_language(state:UPSEState):
    
    prompt = f"evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    
    return {'language_feedback':output.feedback,'individual_scores':[output.score]}

In [51]:
def  evaluate_analysis(state:UPSEState):
    
    prompt = f"evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    
    return {'analysis_feedback':output.feedback,'individual_scores':[output.score]}

In [52]:
def  evaluate_thought(state:UPSEState):
    
    prompt = f"evaluate the clarity of thought of following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_model.invoke(prompt)
    
    return {'clarity_feedback':output.feedback,'individual_scores':[output.score]}

In [53]:
def final_evaluation(state:UPSEState):
    
    #summary feedback 
    prompt = f'based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depht of analysis feedback - {state["analysis_feedback"]} \n clarity feedback - {state["clarity_feedback"]}'
    overall_feedback = model.invoke(prompt).content
    
    #avg feedback 
    avg_score = sum(state["individual_scores"])/len(state["individual_scores"])
    
    return {'overall_feedback':overall_feedback,"avg_score":avg_score}

In [58]:
graph = StateGraph(UPSEState)


graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

#edges 
graph.add_edge(START,"evaluate_language")
graph.add_edge(START,"evaluate_analysis")
graph.add_edge(START,"evaluate_thought")

graph.add_edge("evaluate_language","final_evaluation")
graph.add_edge("evaluate_analysis","final_evaluation")
graph.add_edge("evaluate_thought","final_evaluation")

graph.add_edge("final_evaluation",END)

workflow = graph.compile()


In [57]:
intial_state = {
    'essay' : essay
    
}
workflow.invoke(intial_state)


{'essay': "Social Media\n\nSocial media is very important now days because everyone is using it every time. Peoples are always on phone and they don't do anything without social media. It is good but also bad and this is why it have many effects in the society.\n\nFirst of all social media is helping people for talking with each other. If your friend is in another country you can message him very easy and fastly. Also many peoples posting photos and videos which is entertaining for everyone. Business also use social media because they can selling more products and getting much customers.\n\nBut social media also making many problems. Student waste there time by scrolling reels all day and they not studying properly. Some people become angry or sad because they seeing other peoples expensive life and they thinking there life is very bad. This create depression for many persons.\n\nAnother bad thing is fake news. Everyone can write anything on internet and peoples believe it without chec